# Параметрическое исследование модели SIR на сетях Петри

**Цель:** Исследовать влияние параметров β (коэффициент заражения)
и γ (коэффициент выздоровления) на динамику эпидемии SIR.

**Автор:** Чувакина Мария Владимировна
**Дата:** 2026-04-21

## Теоретическое введение

Модель SIR описывается двумя параметрами:
- **β** — скорость заражения (чем выше, тем быстрее распространяется инфекция)
- **γ** — скорость выздоровления (чем выше, тем быстрее люди выздоравливают)

Базовое репродуктивное число: R₀ = β / γ

При R₀ > 1 эпидемия развивается, при R₀ < 1 — затухает.

## Параметры исследования

In [1]:
using DrWatson
@quickactivate

using Random
using DataFrames, CSV, Plots

include(srcdir("SIRPetri.jl"))
using .SIRPetri

Сетка параметров:
- β ∈ [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
- γ ∈ [0.05, 0.1, 0.15, 0.2]

In [2]:
β_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
γ_values = [0.05, 0.1, 0.15, 0.2]
tmax = 100.0

println("="^60)
println("ПАРАМЕТРИЧЕСКОЕ ИССЛЕДОВАНИЕ МОДЕЛИ SIR")
println("="^60)
println("β = $β_values")
println("γ = $γ_values")
println("Всего комбинаций: $(length(β_values) * length(γ_values))")
println()

ПАРАМЕТРИЧЕСКОЕ ИССЛЕДОВАНИЕ МОДЕЛИ SIR
β = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
γ = [0.05, 0.1, 0.15, 0.2]
Всего комбинаций: 32



## Запуск экспериментов

In [13]:
results = []

for β in β_values
    for γ in γ_values
        println("Исследование β = $β, γ = $γ...")

        net, u0, states = build_sir_network(β, γ)
        df_det = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ])
        Random.seed!(123)
        df_stoch = simulate_stochastic(net, u0, (0.0, tmax), rates = [β, γ])
        peak_I_det = maximum(df_det.I)
        peak_I_stoch = maximum(df_stoch.I)
        final_R_det = df_det.R[end]
        final_R_stoch = df_stoch.R[end]
        peak_time_det = df_det.time[argmax(df_det.I)]

        push!(results, (
            β = β, γ = γ,
            peak_I_det = peak_I_det, peak_I_stoch = peak_I_stoch,
            final_R_det = final_R_det, final_R_stoch = final_R_stoch,
            peak_time_det = peak_time_det,
        ))

        println("    Пик I: $(round(peak_I_det, digits=1)) (детерм.), $(round(peak_I_stoch, digits=1)) (стохаст.)")
    end
end

Исследование β = 0.1, γ = 0.05...
    Пик I: 977.6 (детерм.), 997.0 (стохаст.)
Исследование β = 0.1, γ = 0.1...
    Пик I: 955.6 (детерм.), 995.0 (стохаст.)
Исследование β = 0.1, γ = 0.15...
    Пик I: 934.2 (детерм.), 993.0 (стохаст.)
Исследование β = 0.1, γ = 0.2...
    Пик I: 913.2 (детерм.), 991.0 (стохаст.)
Исследование β = 0.2, γ = 0.05...
    Пик I: 976.4 (детерм.), 999.0 (стохаст.)
Исследование β = 0.2, γ = 0.1...
    Пик I: 953.4 (детерм.), 997.0 (стохаст.)
Исследование β = 0.2, γ = 0.15...
    Пик I: 931.0 (детерм.), 995.0 (стохаст.)
Исследование β = 0.2, γ = 0.2...
    Пик I: 909.0 (детерм.), 995.0 (стохаст.)
Исследование β = 0.3, γ = 0.05...
    Пик I: 976.1 (детерм.), 999.0 (стохаст.)
Исследование β = 0.3, γ = 0.1...
    Пик I: 952.7 (детерм.), 999.0 (стохаст.)
Исследование β = 0.3, γ = 0.15...
    Пик I: 929.9 (детерм.), 997.0 (стохаст.)
Исследование β = 0.3, γ = 0.2...
    Пик I: 907.6 (детерм.), 995.0 (стохаст.)
Исследование β = 0.4, γ = 0.05...
    Пик I: 975.9 (детерм

Детерминированная симуляция

Стохастическая симуляция

## Сохранение результатов

In [14]:
df_results = DataFrame(results)
CSV.write(datadir("sir_parametric_results.csv"), df_results)

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/data/sir_parametric_results.csv"

## Визуализация

### Тепловая карта пика I

In [15]:
p1 = heatmap(
    β_values, γ_values,
    [df_results[df_results.β .== b .&& df_results.γ .== g, :peak_I_det][1] for b in β_values, g in γ_values]',
    xlabel = "β", ylabel = "γ", title = "Peak I (детерм.)",
)
savefig(plotsdir("sir_parametric_heatmap.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_parametric_heatmap.png"

### Зависимость пика I от β

In [16]:
p2 = plot(xlabel = "β", ylabel = "Peak I", title = "Зависимость пика I от β", legend = :topleft)
for γ in γ_values
    df_subset = df_results[df_results.γ .== γ, :]
    plot!(p2, df_subset.β, df_subset.peak_I_det, label = "γ = $γ", marker = :circle)
end
savefig(plotsdir("sir_parametric_beta_dependence.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_parametric_beta_dependence.png"

### Зависимость пика I от γ

In [17]:
p3 = plot(xlabel = "γ", ylabel = "Peak I", title = "Зависимость пика I от γ", legend = :topright)
for β in β_values
    df_subset = df_results[df_results.β .== β, :]
    plot!(p3, df_subset.γ, df_subset.peak_I_det, label = "β = $β", marker = :circle)
end
savefig(plotsdir("sir_parametric_gamma_dependence.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_parametric_gamma_dependence.png"

### Сравнение детерм. и стохаст.

In [18]:
p4 = plot(xlabel = "β", ylabel = "Peak I", title = "Сравнение (γ = 0.1)", legend = :topleft)
df_subset = df_results[df_results.γ .== 0.1, :]
plot!(p4, df_subset.β, df_subset.peak_I_det, label = "Детерминированная", marker = :circle)
plot!(p4, df_subset.β, df_subset.peak_I_stoch, label = "Стохастическая", marker = :square, linestyle = :dash)
savefig(plotsdir("sir_parametric_comparison.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_parametric_comparison.png"

## Выводы

1. **Влияние β:** с ростом β пик заболеваемости увеличивается
2. **Влияние γ:** с ростом γ пик заболеваемости уменьшается
3. **Сравнение методов:** при больших β детерминированная и стохастическая
   динамики близки, при малых β различия заметнее

In [12]:
println()
println("Параметрическое исследование завершено!")


Параметрическое исследование завершено!
